# AQUAVOLTA — Cleaning of the 2022 workbook

Sheets `EXPORT`, `LOCAL`, `DEPENSES`, `SALAIRES` of `DATA FORMATION BI 2022.xlsx`.

**How to run** — install the dependencies once (`pip install -r analytics/requirements.txt`), place the confidential workbook in `data/raw/` (or set `AQUAVOLTA_DATA_DIR`), then run all cells from top to bottom. Each section reloads its own sheet into `res`, so sections can also be run independently after the two setup cells.

The source data is confidential: this notebook is published without outputs. See `analytics/README.md` and `analytics/calculation-audit.md`.

## Setup

In [ ]:
import pandas as pd
import missingno as msno
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Paths and loading helpers (see aquavolta_helpers.py)
from aquavolta_helpers import DATA_DIR, OUTPUT_DIR, load_sheet, save_output

print("Source workbooks :", DATA_DIR)
print("Cleaned outputs  :", OUTPUT_DIR)

## Sheet `EXPORT`

Source: `DATA FORMATION BI 2022.xlsx` (header on row 5).

In [ ]:
res = load_sheet("DATA FORMATION BI 2022.xlsx", sheet_name='EXPORT', header=4)
print(res)

In [ ]:
print(res.shape)

In [ ]:
print(res.size)

In [ ]:
print(res.info())

In [ ]:
print(res.describe())

In [ ]:
print(res.columns)

In [ ]:
msno.bar(res)

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=res, x='PAYS', y='CA €')
plt.xlabel('PAYS')
plt.ylabel('CA €')
plt.title("Boxplot du Chiffre d'Affaires par Pays")
plt.show()

In [ ]:
Q1 = res['CA €'].quantile(0.25)
Q3 = res['CA €'].quantile(0.75)
IQR = Q3 - Q1

# Définir les bornes
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

# Extraire les outliers
outliers = res[(res['CA €'] < borne_inf) | (res['CA €'] > borne_sup)]

print("Nombre d'outliers :", len(outliers))
print(outliers[['PAYS', 'CA €']])

In [ ]:
# Identification des données aberrantes - Z-score
# Fonction pour détecter les outliers en utilisant le z-score
def detect_outliers_zscore(res, column, threshold=2):
    z_scores = (res[column] - res[column].mean()) / res[column].std()
    return res[np.abs(z_scores) > threshold]

# Trouver les outliers pour la colonne 'tva'
outliers = detect_outliers_zscore(res, 'Perte de change')

# Afficher les outliers
print("Outliers détectés:")
print(outliers['Perte de change'])

In [ ]:
count_duplicates = res.duplicated().sum()

print(count_duplicates)

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
save_output(res, "OUTPUT___EXPORT2022.xlsx")

## Sheet `LOCAL`

Source: `DATA FORMATION BI 2022.xlsx` (header on row 5).

In [ ]:
res = load_sheet("DATA FORMATION BI 2022.xlsx", sheet_name='LOCAL', header=4)
print(res)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
print(res.columns)

In [ ]:
msno.bar(res)

In [ ]:
res=res.rename(columns={"date":"type"})
print(res)

In [ ]:
print(res['NOM DU CLIENT'])

> **Review note (audit finding F3).** Missing client names are replaced by the most frequent client. This attributes unidentified invoices to one real client and inflates its revenue in client-level KPIs. Calculation left unchanged; a neutral label such as `CLIENT NON IDENTIFIE` is recommended.

In [ ]:
mode=res[['NOM DU CLIENT']].mode().iloc[0]
for coll in mode.index:
    res[coll]=res[coll].fillna(mode[coll])
print(res.info())

In [ ]:
nan_indices = res['N°'].isna()
count = 1
for i in nan_indices[nan_indices].index:
    res.at[i, 'N°'] = count
    count += 1
res['N°'] = res['N°'].astype(int)
print(res)

In [ ]:
# detect_outliers_zscore() is defined once, in the EXPORT section above
# Trouver les outliers pour la colonne 'tva'
outliers = detect_outliers_zscore(res, 'TTC')

# Afficher les outliers
print("Outliers détectés:")
print(outliers['TTC'])

In [ ]:
# Affichage de l'histogramme initial
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(res['TTC'], bins=30, color='skyblue', edgecolor='black')
plt.title('Données originales')

# Transformation logarithmique des données (en excluant les valeurs négatives)
data_log = np.log(res['TTC'][res['TTC'] > 0])

# Affichage de l'histogramme après transformation logarithmique
plt.subplot(1, 2, 2)
plt.hist(data_log, bins=30, color='salmon', edgecolor='black')
plt.title('Transformation logarithmique')

plt.tight_layout()
plt.show()

In [ ]:
res["Date"] = pd.to_datetime(res["Date"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
save_output(res, "OUTPUT____LOCAL2022.xlsx")

## Sheet `DEPENSES`

Source: `DATA FORMATION BI 2022.xlsx` (header on row 7).

In [ ]:
res = load_sheet("DATA FORMATION BI 2022.xlsx", sheet_name='DEPENSES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
msno.bar(res)

In [ ]:
count_duplicates = res.duplicated().sum()

print(count_duplicates)

In [ ]:
print(res.info())

In [ ]:
res["DATE"] = pd.to_datetime(res["DATE"]).dt.strftime("%d/%m/%Y")
print(res)

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
save_output(res, "OUTPUT___DEPENSE___2022.xlsx")

## Sheet `SALAIRES`

Source: `DATA FORMATION BI 2022.xlsx` (header on row 7).

In [ ]:
res = load_sheet("DATA FORMATION BI 2022.xlsx", sheet_name='SALAIRES', header=6)
print(res)

In [ ]:
print(res.columns)

In [ ]:
print(res.shape)

In [ ]:
print(res.info())

In [ ]:
# detect_outliers_zscore() is defined once, in the EXPORT section above
# Trouver les outliers pour la colonne 'tva'
outliers = detect_outliers_zscore(res, 'COUT TOTAL')

# Afficher les outliers
print("Outliers détectés:")
print(outliers['COUT TOTAL'])

In [ ]:
count_duplicates = res.duplicated().sum()

print(count_duplicates)

In [ ]:
mois_map = {
    "JANVIER": "01",
    "FEVRIER": "02",
    "MARS": "03",
    "AVRIL": "04",
    "MAI": "05",
    "JUIN": "06",
    "JUILLET": "07",
    "AOUT": "08",
    "SEPTEMBRE": "09",
    "OCTOBRE": "10",
    "NOVEMBRE": "11",
    "DECEMBRE": "12"
}
res["date"] = res["MOIS"].map(mois_map)  
res["date"] = "01/" + res["date"] + "/2022"
print(res[["MOIS", "date"]].head())

In [ ]:
res["DEPARTEMENT"] = res["DEPARTEMENT"].replace({
    "ELEC": "ELECTRICITE",
    "GCIVILE": "GENI CIVIL"
})
print(res)

In [ ]:
save_output(res, "OUTPUT____SALAIRE2022.xlsx")